In [2]:
# Standard libraries we will need
import math
from collections import defaultdict

print("Libraries imported successfully.")

Libraries imported successfully.


# Understanding the PageRank Algorithm
### A Beginner's Step-by-Step Guide

**Author:** *Anton Popov*  
**Course:** Math Concepts for Developers - Softuni  
**Date:** March 2026

---

## 1. Why I Chose This Topic

I use Google every day, and I never really thought about how it works. I decided to dig into the PageRank algorithm - the original idea behind how Google ranks web pages - and try to understand it from the ground up.

What I found was that the core idea is actually quite elegant and not as scary as it sounds. You do not need to be a professional engineer to follow it. That is what this notebook is about: walking through PageRank step by step, explaining each piece before jumping into the math or the code.

By the end, I want anyone reading this to be able to answer the question: *how does Google decide which page goes first?*

---
## 2. What Is PageRank?

### 2.1 The problem with the old way

Before Google, search engines ranked pages by counting how many times your search term appeared on the page. If you searched for "best pizza recipe" and a page said "pizza" 100 times, it would rank highly - even if it was a terrible page full of spam.

Larry Page and Sergey Brin, two PhD students at Stanford in 1998, thought there had to be a better way. Their idea was inspired by how academic research works: an important research paper is one that many other important papers cite. The same logic should apply to web pages - an important page is one that many other important pages link to.

The algorithm they built to do this is called **PageRank** (named after Larry Page).

### 2.2 The key insight - links as votes

The central idea of PageRank is simple:

> A link from one page to another is like a vote of confidence. Pages with more votes are more important. But votes from important pages count for more than votes from unimportant ones.

Here is a small example to make this concrete. Imagine we have just five web pages:

- Page **A** is linked to by pages B, C, and D
- Page **B** is linked to by page A
- Page **C** is linked to by page A
- Page **D** is only linked to by page E, which is an obscure page nobody links to
- Page **E** is linked to by nobody

Even though A and D both receive links, A is more important - because the pages linking to A are themselves well-connected. PageRank captures this difference.

### 2.3 The random surfer - an intuitive model

There is a very helpful way to think about PageRank without any math at all. Imagine someone browsing the internet completely at random:

- They start on some random page
- At each step, they either click a random link on the current page, or - when they get bored - they type a completely new address and jump somewhere random
- They keep doing this forever

After a very long time, some pages will be visited much more often than others. Those are the important pages. The **PageRank score** of a page is simply the fraction of time this imaginary random surfer spends there.

The probability that the surfer follows a link (rather than teleporting) is called the **damping factor**, written as $d$. Google originally set this to $0.85$, which means 85% of the time the surfer follows a link, and 15% of the time they teleport to a random page. We will use the same value throughout this notebook.

The teleportation step is important for a practical reason: some pages have no outgoing links at all. Without teleportation, our random surfer would get stuck on such a page forever. Teleportation ensures they can always escape.

---
## 3. The Math - Explained Simply

This section introduces the mathematics behind PageRank. I will explain each concept in plain language first, and then show the formula.

### 3.1 Representing the web as a graph

In mathematics, a **directed graph** is a way of representing relationships that have a direction. Each web page is a **node**, and each hyperlink is a **directed edge** - an arrow pointing from the page containing the link to the page it points to.

We write a graph as $G = (V, E)$ where $V$ is the set of all nodes (pages) and $E$ is the set of all directed edges (links).

We can describe the connections as a matrix. The **adjacency matrix** $A$ has a 1 where a link exists and a 0 where it does not:

$$A_{ij} = \begin{cases} 1 & \text{if page } i \text{ links to page } j \\ 0 & \text{otherwise} \end{cases}$$

The number of links going *out* from a page is called the **out-degree**, written $d_i^{\text{out}}$:

$$d_i^{\text{out}} = \sum_{j} A_{ij}$$

### 3.2 The PageRank formula

When a page links to others, it shares its importance equally among all of them. If page B has a rank of 0.4 and links to 4 other pages, it passes $\frac{0.4}{4} = 0.1$ to each of them. This fraction is written as $\frac{r_i}{d_i^{\text{out}}}$.

Putting the teleportation model and the link-sharing idea together, the PageRank score of any page $j$ is:

$$r_j = \frac{1 - d}{n} + d \sum_{i \,:\, i \to j} \frac{r_i}{d_i^{\text{out}}}$$

Let's break this down piece by piece:

- $r_j$ is the PageRank score we want to find for page $j$
- $d = 0.85$ is the damping factor
- $n$ is the total number of pages
- $\frac{1 - d}{n}$ is a small baseline score every page gets from teleportation
- The sum adds up contributions from every page $i$ that has a link pointing to $j$
- Each contributing page $i$ donates $\frac{r_i}{d_i^{\text{out}}}$ - its rank divided by how many links it has

Notice the challenge: to compute the rank of page $j$, we need the ranks of all pages that link to it - but those ranks depend on the rank of $j$. Everything is circular. The trick is to solve this iteratively.

### 3.3 The Google Matrix

All of the above can be written more compactly using a matrix called the **Google Matrix** $G$:

$$G = d \cdot M^T + (1 - d) \cdot \frac{1}{n} \cdot \mathbf{1}$$

where $M$ is the transition matrix (each row divided by the page's out-degree) and $\frac{1}{n} \cdot \mathbf{1}$ represents the uniform teleportation. Every column of $G$ sums to exactly 1.0, which means our PageRank scores will always sum to 1 as well.

The PageRank vector $\mathbf{r}$ is the solution to:

$$G \cdot \mathbf{r} = \mathbf{r}$$

A vector that satisfies this is called an **eigenvector** with eigenvalue 1. The **Perron–Frobenius theorem** guarantees this solution always exists and is unique for a matrix like $G$.

### 3.4 How we find the answer - Power Iteration

Finding the eigenvector directly would require inverting a huge matrix - not practical for billions of pages. Instead we use **Power Iteration**:

1. Start by giving every page the same score: $r_j = \frac{1}{n}$
2. Apply the PageRank formula to update every page's score
3. Repeat step 2 over and over
4. Stop when the scores barely change between steps

It can be shown that the error after $t$ steps is at most $d^t$. With $d = 0.85$, after just 50 steps the error is $0.85^{50} \approx 0.0003$ - accurate enough for most purposes.

---
## 4. Building the Algorithm in Python

Now let's write this in Python.

### 4.1 Step 1 - Represent the graph

The first thing we need is a way to store the graph. We represent it as a list of links - each link is a pair `(from_page, to_page)`. From this list, we build two dictionaries:

- `out_links`: for each page, which pages does it link *to*?
- `in_links`: for each page, which pages link *to it*?

The `in_links` dictionary is the one we will use in the PageRank formula - for each page $j$, we need to know which pages $i$ are sending rank to it.

In [ ]:
def build_graph(edges):
    """
    Takes a list of (from_page, to_page) pairs and builds
    a graph dictionary we can use for PageRank.
    """
    out_links = defaultdict(list)  # page -> list of pages it links TO
    in_links  = defaultdict(list)  # page -> list of pages that link TO it
    all_pages = set()

    for from_page, to_page in edges:
        out_links[from_page].append(to_page)
        in_links[to_page].append(from_page)
        all_pages.add(from_page)
        all_pages.add(to_page)

    pages = sorted(all_pages)

    return {
        'pages': pages,
        'out_links': dict(out_links),
        'in_links': dict(in_links),
        'n': len(pages)
    }

# Let's test it with a tiny example
tiny_edges = [
    ('A', 'B'),
    ('A', 'C'),
    ('B', 'C'),
    ('C', 'A'),
]

tiny_graph = build_graph(tiny_edges)

print("Pages found:   ", tiny_graph['pages'])
print("Out-links:     ", dict(tiny_graph['out_links']))
print("In-links:      ", dict(tiny_graph['in_links']))

Pages found:    ['A', 'B', 'C']
Out-links:      {'A': ['B', 'C'], 'B': ['C'], 'C': ['A']}
In-links:       {'B': ['A'], 'C': ['A', 'B'], 'A': ['C']}
